In [6]:
import pandas as pd

In [26]:
cb = pd.read_csv('customer_shopping_behavior.csv')
cb.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [27]:
cb.isnull().sum() #we have to find null first

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [30]:
g1=cb.groupby('Category')
g1["Purchase Amount (USD)"].median("Purchase Amount (USD)") #to fill the null value by category wise purchase

Category
Accessories    60.0
Clothing       60.0
Footwear       60.0
Outerwear      54.5
Name: Purchase Amount (USD), dtype: float64

In [31]:
cb['Review Rating']=g1['Review Rating'].transform(lambda x:x.fillna(x.median())) #to fill the null using lanbda,fillna

In [32]:
cblower=cb.columns.str.lower()                      #to change the column name in snakecase format
cblower=cblower.str.replace(" ","_")
cb.columns=cblower
cb=cb.rename(columns={"purchase_amount_(usd)":"purchase_amount"})
cb.head()

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [33]:
# res=[]
# for i in cb['age']:
#     if i<=30:                                            #old way to add new column by age category
#         res.append('young adult')
#     elif i<=40:
#         res.append('adult')
#     elif i<=50:
#         res.append('middle-age')
#     else:
#         res.append('senior')
# cb['age_group']=res

names=['young','adult','middle-aged','senior']
cb['age_group']=pd.qcut(cb['age'],q=4,labels=names)
cb[['age_group','age']].head(10)

,age_group,age
0,middle-aged,55
1,young,19
2,middle-aged,50
3,young,21
4,middle-aged,45
5,middle-aged,46
6,senior,63
7,young,27
8,young,26
9,middle-aged,57


In [34]:
# create column purchase_frequency_days
cb['frequency_of_purchases'].unique()
frequency_mapping = {
'Fortnightly' : 14,'Weekly' : 7,'Monthly': 30,'Quarterly' : 90,'Bi-Weekly': 14,'Annually': 365,'Every 3 Months' : 90}
cb['frequency_of_purchases_days']=cb['frequency_of_purchases'].map(frequency_mapping)
cb.head()


,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases,age_group,frequency_of_purchases_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly,middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly,young,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly,middle-aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly,young,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually,middle-aged,365


In [35]:
#
(cb['discount_applied']==cb['promo_code_used']).all()   #it's used to check the two columns have same values
cb=cb.drop('promo_code_used',axis=1)                    #if it's have same column we have to drop one

In [15]:
# code upload the data into database for mysql

In [12]:
pip install pymysql sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [36]:
from sqlalchemy import create_engine
# MySQL connection
username = "root"
password = "root"
host = "localhost"
port = "3306"
database = "project_1"
engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")
# Write Data to MySQL
table_name = "customer_behavior" # choose any table name
cb.to_sql(table_name, engine, if_exists="replace", index=False)
# Read back sample
df=pd.read_sql("SELECT * FROM customer_behavior LIMIT 5;", engine)

In [38]:
df

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,frequency_of_purchases_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,young,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,middle-aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,young,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,middle-aged,365
